# University locations

Locates universities relative to speaker regions, used in the socio-economic
features.

In [ ]:
import requests 
import pandas as pd 
from bs4 import BeautifulSoup
import time
import json
from urllib.parse import urlparse, parse_qs
import pickle
import os
import glob

In [ ]:
def scrape_university_data(base_url="https://www.4icu.org/reviews/index", pages=27):
    # Initialize empty lists to store the data
    universities = []
    countries = []
    links = []
    
    # Loop through all the pages (from index2.htm to index27.htm)
    for i in range(2, pages + 1):
        # Construct the page URL
        url = f"{base_url}{i}.htm"
        response = requests.get(url)
        
        # Check if the request was successful
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            # Locate the table with the university data
            table_rows = soup.select("tbody tr")
            
            # Iterate over the rows in the table
            for row in table_rows:
                country_tag = row.find('img')
                university_tag = row.find('a', class_='lead')
                
                # If both elements are found, extract data
                if country_tag and university_tag:
                    country = country_tag['alt']
                    name = university_tag.get_text(strip=True)
                    href = "https://www.4icu.org" + university_tag['href']
                    
                    # Append to lists
                    countries.append(country)
                    universities.append(name)
                    links.append(href)
        else:
            print(f"Failed to retrieve data from {url}")
    
    # Create a DataFrame with the scraped data
    df = pd.DataFrame({
        'Country': countries,
        'University Name': universities,
        'URL': links
    })
    
    return df

# Example usage:
df_universities = scrape_university_data()
print(df_universities)

In [ ]:
df_universities

In [ ]:
def get_location_map_link(df, start_index=0):
    # Initialize a list to store the Location Map URLs
    location_links = []
    
    for url in df['URL']:
        # Request the university's page
        response = requests.get(url)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            # Find the "Location Map and Satellite View" link
            location_tag = soup.find('a', href=lambda x: x and "maps" in x)
            location_link = location_tag['href'] if location_tag else None
            print(location_link)
            location_links.append(location_link)
        else:
            location_links.append(None)
        
        time.sleep(1)  # Adding a delay to avoid overloading the server
    
    # Add the location map links as a new column in the DataFrame
    df['Location Map URL'] = location_links
    return df

In [ ]:
df_universities_with_location = get_location_map_link(df_universities)
print(df_universities_with_location)

In [ ]:
with open('Untitled-1.txt', 'r') as f:
    data = f.readlines()

In [ ]:
cleaned_links = [link.strip() for link in data]

In [ ]:
df_universities['Location Map URL'] = cleaned_links

In [ ]:
df_universities.to_pickle('unis_locaysh_link.pkl')

In [ ]:
# Function to extract latitude and longitude from the iframe src
def extract_lat_long(iframe_src):
    if 'location=' in iframe_src:
        coords = iframe_src.split('location=')[1].split('&')[0]
        latitude, longitude = coords.split(',')
        return latitude, longitude
    return None, None

# Function to get the links from the given university URL
def get_links(university_url):
    response = requests.get(university_url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        # Find all iframes and extract their src attributes
        iframes = soup.find_all('iframe')
        links = [iframe['src'] for iframe in iframes if 'src' in iframe.attrs and 'location=' in iframe['src']]
        if links:
            last_link = links[-1]  # Get the last link with 'location='
            print(last_link)
            return last_link
        else:
            print("No link with 'location=' found.")
            return None
    else:
        print(f"Failed to retrieve {university_url} with status code {response.status_code}")
        return []

# Main function to update DataFrame with latitude and longitude
def add_lat_long_to_dataframe(df, university_url_column):
    latitudes = []
    longitudes = []
    
    for url in df[university_url_column]:
        links = get_links(url)  # Get the links from the university page
        if links:
            for link in links:
                lat, long = extract_lat_long(link)
                latitudes.append(lat)
                longitudes.append(long)
        else:
            latitudes.append(None)
            longitudes.append(None)
    
    df['latitude'] = latitudes
    df['longitude'] = longitudes

In [ ]:
df = df_universities[0:10]

In [ ]:
df

In [ ]:
add_lat_long_to_dataframe(df, 'Location Map URL')

In [ ]:
def extract_lat_long(iframe_src):
    if 'location=' in iframe_src:
        coords = iframe_src.split('location=')[1].split('&')[0]
        latitude, longitude = coords.split(',')
        return latitude, longitude
    return None, None

# Function to get the last link containing 'location='
def get_last_location_link(university_url):
    response = requests.get(university_url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        iframes = soup.find_all('iframe')
        links = [iframe['src'] for iframe in iframes if 'src' in iframe.attrs and 'location=' in iframe['src']]
        if links:
            return links[-1]  # Return only the last link with 'location='
    print(f"No location link found for {university_url}")
    return None

# Main function to update DataFrame with latitude and longitude
def add_lat_long_to_dataframe(df, university_url_column):
    latitudes = []
    longitudes = []
    
    for url in df[university_url_column]:
        link = get_last_location_link(url)  # Get the last link with 'location='
        if link:
            lat, long = extract_lat_long(link)
            latitudes.append(lat)
            longitudes.append(long)
        else:
            latitudes.append(None)
            longitudes.append(None)
    
    # Ensure that latitudes and longitudes lists are of the same length as the DataFrame
    df['latitude'] = latitudes
    df['longitude'] = longitudes

In [ ]:
add_lat_long_to_dataframe(df, 'Location Map URL')

In [ ]:
df

In [ ]:
def get_last_location_link(university_url):
    response = requests.get(university_url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        iframes = soup.find_all('iframe')
        links = [iframe['src'] for iframe in iframes if 'src' in iframe.attrs and 'location=' in iframe['src']]
        if links:
            return links[-1]
    return None

# Extract latitude and longitude from iframe src URL
def extract_lat_long(iframe_src):
    if 'location=' in iframe_src:
        coords = iframe_src.split('location=')[1].split('&')[0]
        latitude, longitude = coords.split(',')
        return latitude, longitude
    return None, None

# Process each chunk of the DataFrame
def process_and_save_chunk(df_chunk, chunk_number, university_url_column):
    latitudes = []
    longitudes = []
    
    for url in df_chunk[university_url_column]:
        link = get_last_location_link(url)
        if link:
            lat, long = extract_lat_long(link)
            latitudes.append(lat)
            longitudes.append(long)
        else:
            latitudes.append(None)
            longitudes.append(None)
    
    # Add latitude and longitude columns to the chunk
    df_chunk['latitude'] = latitudes
    df_chunk['longitude'] = longitudes
    
    # Save chunk to a pickle file
    filename = f"university_data_chunk_{chunk_number}.pkl"
    with open(filename, 'wb') as f:
        pickle.dump(df_chunk, f)
    print(f"Chunk {chunk_number} processed and saved as {filename}")

# Main function to split DataFrame and process in chunks
def process_large_dataframe(df, university_url_column, chunk_size=100):
    num_chunks = (len(df) // chunk_size) + 1  # Calculate number of chunks
    
    for chunk_number in range(num_chunks):
        # Get the current chunk of the DataFrame
        start_idx = chunk_number * chunk_size
        end_idx = min((chunk_number + 1) * chunk_size, len(df))
        df_chunk = df.iloc[start_idx:end_idx].copy()  # Copy to avoid modifying original df

        # Process and save the chunk
        process_and_save_chunk(df_chunk, chunk_number + 1, university_url_column)

In [ ]:
process_large_dataframe(df_universities, 'Location Map URL', chunk_size=100)

In [ ]:
!pwd

In [ ]:
directory_path = 'data/universities'

In [ ]:
file_pattern = os.path.join(directory_path, "university_data_chunk_*.pkl")
pickle_files = glob.glob(file_pattern)

# Load each pickle file and append to a list
dataframes = []
for file in pickle_files:
    df_chunk = pd.read_pickle(file)
    dataframes.append(df_chunk)
    print(f"Loaded {file}")

# Concatenate all DataFrames into one
full_dataframe = pd.concat(dataframes, ignore_index=True)
print("All chunks concatenated successfully.")

In [ ]:
full_dataframe

In [ ]:
df_universities

In [ ]:
df_universities_3 = df_universities[7000::]

In [ ]:
df_universities_3

In [ ]:
process_large_dataframe(df_universities_3, 'Location Map URL', chunk_size=100)

In [ ]:
file_pattern = os.path.join(directory_path, "university_data_chunk_*.pkl")
pickle_files = glob.glob(file_pattern)

# Load each pickle file and append to a list
dataframes = []
for file in pickle_files:
    df_chunk = pd.read_pickle(file)
    dataframes.append(df_chunk)
    print(f"Loaded {file}")

# Concatenate all DataFrames into one
full_dataframe2 = pd.concat(dataframes, ignore_index=True)
print("All chunks concatenated successfully.")

In [ ]:
full_dataframe2

In [ ]:
full_dataframe

In [ ]:
universities_data = pd.concat([full_dataframe, full_dataframe2], axis=0, ignore_index=True)

In [ ]:
universities_data

In [ ]:
universities_data.to_csv('all_universities_locations.csv')

In [ ]:
pwd